# Image Editing Tools

Exercise every registered tool using lightweight local examples and opt-in model-backed examples.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file())
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from IPython.display import display
from PIL import Image, ImageDraw
from tools import image_editing_tools

tools_by_name = {tool.name: tool for tool in image_editing_tools()}
expected_tools = {
    'apply_filter', 'edit_object', 'flip_object', 'generate_object',
    'move_object', 'remove_object',
    'replace_object', 'rotate_object', 'scale_object', 'segment_object', 'shift_object',
}
assert set(tools_by_name) == expected_tools

def call_tool(name, **inputs):
    return tools_by_name[name].forward(**inputs)

RUN_MODEL_BACKED = False
image = Image.open(PROJECT_ROOT / 'resources/test_images/sample1.jpg').convert('RGB')
landscape = Image.open(PROJECT_ROOT / 'resources/test_images/sample2.jpg').convert('RGB')
display(image, landscape)
print(f'{len(tools_by_name)} tools loaded; model-backed examples enabled: {RUN_MODEL_BACKED}')

## Prepare reusable example inputs

The ellipse is a hand-drawn selection used for local compositing examples.

In [ ]:
selection_mask = Image.new('L', image.size, 0)
ImageDraw.Draw(selection_mask).ellipse((
    round(image.width * 0.22), round(image.height * 0.18),
    round(image.width * 0.78), round(image.height * 0.82),
), fill=255)
selection_box = list(selection_mask.getbbox())
display(selection_mask)
print('selection box:', selection_box)

In [ ]:
display(call_tool('apply_filter', image=image, filter_name='saturation', value=0.5))

## Model-backed tools

Set `RUN_MODEL_BACKED = True` in the setup cell to run these calls. Model weights may download on first use.

In [ ]:
if RUN_MODEL_BACKED:
    display(call_tool('segment_object', image=image, point=[image.width // 2, image.height // 2]))
    display(call_tool('segment_object', image=image, box=selection_box))
    display(call_tool('segment_object', image=image, scribble=[[image.width // 3, image.height // 2], [image.width // 2, image.height // 2]]))

In [ ]:
if RUN_MODEL_BACKED:
    text_mask = call_tool('segment_object', image=image, prompt='cat')
    display(text_mask)

In [ ]:
if RUN_MODEL_BACKED:
    display(call_tool('remove_object', image=image, mask=selection_mask))

In [ ]:
if RUN_MODEL_BACKED:
    generated = call_tool('generate_object', image=image, prompt='a calm grassy garden', mask=selection_mask)
    display(generated)

In [ ]:
if RUN_MODEL_BACKED:
    display(call_tool('generate_object', image=image, prompt='a natural photo', box=selection_box, phrase='a red ball'))

In [ ]:
if RUN_MODEL_BACKED:
    display(call_tool('edit_object', image=image, mask=selection_mask, prompt='make the selected area warmer in color', mode='local'))

In [ ]:
if RUN_MODEL_BACKED:
    replacement = landscape.crop((0, 0, landscape.width // 2, landscape.height // 2))
    display(call_tool('replace_object', image=image, mask=selection_mask, replacement=replacement))

In [ ]:
if RUN_MODEL_BACKED:
    display(call_tool('move_object', image=image, mask=selection_mask, target=[image.width * 0.72, image.height * 0.5]))

In [ ]:
if RUN_MODEL_BACKED:
    display(call_tool('scale_object', image=image, mask=selection_mask, factor=1.25))

In [ ]:
if RUN_MODEL_BACKED:
    display(call_tool('flip_object', image=image, mask=selection_mask, axis='x'))

In [ ]:
if RUN_MODEL_BACKED:
    display(call_tool('rotate_object', image=image, mask=selection_mask, angle=20))

In [ ]:
if RUN_MODEL_BACKED:
    display(call_tool('shift_object', image=image, mask=selection_mask, offset=[30, 20]))